# Time-series walkthrough: watching over a power transformer

`kcai-data-sampling` turns recorded samples into new, fully described samples: a **transformation** `T : x ↦ x′` is one algorithm with its resolved parameters and seed, and every output carries a record of what was done to which input. For time series, the sample is a **window** cut out of a long record. This notebook shows the interface on a real dataset, then what it is for.

A grid operator runs power transformers. Each one logs its **load** and its **oil temperature** (`OT`) every hour, and the operator relies on two models:

- a **forecaster** that predicts the oil temperature of the next 24 hours, to anticipate overheating;
- an **anomaly detector** that must raise an alert when a transformer misbehaves (cooling degradation, cooling cycling, acquisition glitches), and stay silent otherwise.

The recorded data is real but narrow: two transformers, two years, **no failure at all**. The transformations of `kcai-data-sampling-timeseries` create the situations nobody recorded, and the notebook uses them to answer two questions:

1. **Forecasting**: how much worse does the forecast get when the history is degraded (noisy probe, coarser logging)?
2. **Anomaly detection**: which faults does the detector catch, from what size, and does it stay quiet on harmless variability?

**How this notebook works.** Everything is computed here, step by step, with the package's own objects:

1. the **loader** cuts the series into one-week windows, the samples, each with a **key** that says where it comes from (sections 1–2);
2. the **transformation classes** turn a week into a new week, and say what they did (sections 3–6);
3. a list of transformations, with swept parameters, turns the record into **test sets** (section 7);
4. the two models are evaluated on them (sections 8–9);
5. at the end, the same test sets are produced by the **pipeline** from the YAML configs of `examples/config/`, which is how the package is used outside a notebook, and we check that they are the same windows (section 10).

**Data.** [ETT](https://github.com/zhouhaoyi/ETDataset) (*Electricity Transformer Temperature*, Zhou et al., Informer, AAAI 2021), the reference benchmark of forecasting: `ETTh1` and `ETTh2`, hourly, July 2016 to June 2018. ETT is licensed **CC BY-ND 4.0**: the data is downloaded by `scripts/fetch_ett_sample.py` (the setup cell runs it if needed), and the outputs of this notebook (generated windows, figures) are adapted material, **not to be committed or shared**. The notebook is committed without its outputs.

**Models.** Deliberately simple stand-ins for the operator's models, in `helpers/ett_models.py` next to this notebook. They need scikit-learn, which is not a workspace dependency. From the repository root:

```bash
uv run --with scikit-learn --with matplotlib --with ipykernel --with jupyterlab \
    jupyter lab examples/notebooks/timeseries-walkthrough.ipynb
```

Seed 0 everywhere.

## Setup

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch, Polygon, Rectangle
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import yaml

%config InlineBackend.figure_format = 'png'
pd.set_option("display.max_colwidth", None)

# Work from the repository root: every path in the configs is relative to it.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(REPO)
sys.path.insert(0, str(REPO / "examples" / "notebooks"))

# The package's objects: the loader that cuts windows, the transformations, the sweeps, the pipeline.
from kcai_data_sampling_core.models.sweep import SweepConfig
from kcai_data_sampling_job.cli import run
from kcai_data_sampling_job.dataloaders.api.timeseries import TimeSeriesLoaderConfig, TimeSeriesParquetDataLoader
from kcai_data_sampling_timeseries.api.transformations import (
    AddSinusoid,
    ColoredNoise,
    Drift,
    GarchNoise,
    ImpulseNoise,
    Jitter,
    LowPass,
    OUNoise,
)

from helpers import ett_models as em

DATA = Path("examples/data/ett_sample/series.parquet")
if not DATA.exists():
    subprocess.run([sys.executable, "scripts/fetch_ett_sample.py"], check=True)

## 1. The data

The data file is a parquet table with one row per transformer. Each row holds a `(T, 7)` array: T hours, seven channels, six loads (`HUFL`, `HULL`, `MUFL`, `MULL`, `LUFL`, `LULL`: high, middle and low, useful and useless) and the oil temperature `OT`, channel 6.

In [ ]:
table = pq.read_table(DATA).to_pandas()
table.assign(values=table["values"].map(lambda v: f"({len(v)}, {len(v[0])}): first hour {np.round(v[0], 1).tolist()}"))

This notebook follows the oil temperature alone. Everything the models learn comes from the first 70 % of each series; every result below is measured on windows that start after it, so no evaluated window overlaps the training data.

In [ ]:
SERIES = em.load_ot(DATA)  # {series_id: (T,) oil temperature}
ENDS = em.train_end(SERIES)  # first evaluation hour, per series
START = pd.Timestamp(table.start[0])

fig, axes = plt.subplots(2, 1, figsize=(12.2, 6.2), sharex=True, sharey=True)
for ax, (sid, ot) in zip(axes, SERIES.items()):
    t = pd.date_range(START, periods=len(ot), freq="h")
    ax.plot(t, ot, lw=0.5)
    ax.axvspan(t[ENDS[sid]], t[-1], color="gray", alpha=0.15, lw=0)
    ax.text(t[ENDS[sid]], 0.95, "  evaluation", transform=ax.get_xaxis_transform(), va="top", fontsize=13)
    ax.set_title(f"{sid}: oil temperature (°C)", loc="left")
fig.tight_layout()
plt.show()

## 2. Samples are cut, not given

In image classification a sample is given: one image, collected as such, with an annotation outside the pixels. A time series is one long record; **the samples are windows cut out of it**, and the cut is a choice.

The loader makes that choice explicit. Its configuration is the same block as in the YAML configs (`dataloaders.loaders`): which file, which column and channels, and how to cut. Here, the oil temperature only (`channels: [6]`), windows of **one week (168 hours)**, **one starting every day** (stride 24).

In [ ]:
LOADER = {
    "name": "ett_ot",
    "type": "timeseries_parquet",
    "path": "examples/data/ett_sample/series.parquet",
    "id_column": "series_id",
    "values_column": "values",
    "sampling_rate": 0.000277778,  # 1 / 3600 Hz, informative: parameters are per step (per hour)
    "channels": [6],  # OT only
    "load_batch_size": 64,
    "windows": {"mode": "consecutive", "window": 168, "stride": 24},
}


def select(loader):
    """Cut the series into windows: the loader's selection, as a list of batches."""
    config = TimeSeriesLoaderConfig.model_validate(loader)
    (selection,) = TimeSeriesParquetDataLoader(config.name, config).get_selections()
    selection.bootstrap(None)
    return list(selection)


def keys_of(batches):
    """Where every window comes from: one row per window, with its id."""
    keys = pd.concat([b.columns.to_pandas().assign(id=b.ids) for b in batches], ignore_index=True)
    return keys[["series_id", "start_window", "end_window", "id"]]


def window_at(batches, series_id, start):
    """The one-window batch of ``series_id`` that starts at hour ``start``."""
    for batch in batches:
        cols = batch.columns.to_pandas()
        hit = np.flatnonzero((cols.series_id == series_id) & (cols.start_window == start))
        if len(hit):
            return batch.row(int(hit[0]))
    raise KeyError((series_id, start))


BATCHES = select(LOADER)
KEYS = keys_of(BATCHES)
print(f"{len(KEYS)} windows in {len(BATCHES)} batches; a batch holds a (windows, hours, channels) array: {BATCHES[0].data.shape}")
KEYS.groupby("series_id").size().to_frame("windows")

Two series of 17,420 hours give 1,438 one-week windows. They **overlap** (two neighbours share six days) and they are **ordered in time**, so the split between training and evaluation is made in time, never at random. Each window has an id, derived from where it was cut: every sample generated from it will point back to it.

**The key.** Each window carries its key: `series_id`, `start_window` and `end_window` (inclusive, in steps of its series) and, when the loader is given a clock, `t0_series`. Every relation between two windows is computed from it, never stored:

- **same series**: equal `series_id`;
- **overlap**, same series: `max(0, min(e₁, e₂) − max(s₁, s₂) + 1)` shared steps;
- **order and lag**, same series: when `s₂ > e₁`, window 1 precedes window 2 by `s₂ − e₁ − 1` steps;
- **simultaneity**, two series: the clock intervals `[t0_series + s, t0_series + e]` intersect.

The clock is read from an integer column of steps, the loader's `t0_column`. ETT gives the timestamp of each series' first step, `start`: we compute `t0` from it, in hours from the earliest series, write it in a copy of the table, and pass it to the loader. Both transformers start at the same hour, so both get `t0 = 0`: their windows that start at the same hour are simultaneous. The ids do not change: an id depends only on `(series, start, end)`.

In [ ]:
starts = pd.to_datetime(table.start)
T0 = dict(zip(table.series_id, ((starts - starts.min()) // pd.Timedelta(hours=1)).tolist()))
print("t0 (hours from the earliest series):", T0)
T0_PATH = DATA.with_name("series_t0.parquet")
pq.write_table(pq.read_table(DATA).append_column("t0", pa.array([T0[s] for s in table.series_id], pa.int64())), T0_PATH)

CLOCKED = pd.concat(
    [b.columns.to_pandas().assign(id=b.ids) for b in select({**LOADER, "path": str(T0_PATH), "t0_column": "t0"})],
    ignore_index=True,
)[["series_id", "start_window", "end_window", "t0_series", "id"]]
PER_SERIES = int((CLOCKED.series_id == "ETTh1").sum())
print("same windows, same ids:", CLOCKED.id.equals(KEYS.id))
CLOCKED.iloc[[0, 1, 7, 8, PER_SERIES, PER_SERIES + 1]]

With one window a day, window 1 of `ETTh1` shares six days with window 0, window 7 starts right after it (lag 0), window 8 one day later. The first window of `ETTh2` covers the same hours as window 0 of `ETTh1`, six days of window 1, and none of window 7.

In [ ]:
def shared_steps(s1, e1, s2, e2):
    return max(0, min(e1, e2) - max(s1, s2) + 1)


def relation(u, v):
    if u.series_id == v.series_id:
        shared = shared_steps(u.start_window, u.end_window, v.start_window, v.end_window)
        if shared:
            return f"same series, overlap of {shared} steps"
        return f"same series, lag of {v.start_window - u.end_window - 1} steps"
    clock = shared_steps(
        u.t0_series + u.start_window, u.t0_series + u.end_window, v.t0_series + v.start_window, v.t0_series + v.end_window
    )
    return f"two series, simultaneous on {clock} steps" if clock else "two series, not simultaneous"


rows = dict(enumerate(CLOCKED.itertuples()))
for i, j in [(0, 1), (0, 7), (0, 8), (0, PER_SERIES), (1, PER_SERIES), (7, PER_SERIES)]:
    print(f"window {i:>3} ({rows[i].series_id}) vs window {j:>3} ({rows[j].series_id}): {relation(rows[i], rows[j])}")

**Independent windows.** Instead of cutting at a fixed stride, the loader can draw `n_windows` positions per series, seeded by `seed` and the series id; with `non_overlap: true`, no two windows of a series share a step. The same seed draws the same windows. A window gets the same id whichever mode cut it.

In [ ]:
independent = {"mode": "independent", "window": 168, "n_windows": 3, "non_overlap": True, "seed": 0}
drawn = keys_of(select({**LOADER, "windows": independent}))
again = keys_of(select({**LOADER, "windows": independent}))
other = keys_of(select({**LOADER, "windows": {**independent, "seed": 1}}))
print("same seed, same windows:", drawn.equals(again))
print("other seed, other windows:", not drawn.start_window.equals(other.start_window))
drawn

**Channels are chosen by the data selection.** A transformation transforms every channel of the windows it receives; which channels a window holds is the loader's `channels`. The demo keeps the oil temperature (`channels: [6]`). A second selection with `channels: [0]` holds the high useful load `HUFL` instead: the same windows, with the same ids, one channel each. Section 4 uses both.

In [ ]:
LOAD_BATCHES = select({**LOADER, "channels": [0]})
print("same windows, same ids:", keys_of(LOAD_BATCHES).equals(KEYS))
print("a load window:", LOAD_BATCHES[0].data.shape, "; a temperature window:", BATCHES[0].data.shape)

We follow one window through the notebook: a spring week of `ETTh2` starting on 2 April 2018 (hour 15,360), in the evaluation part, with a clear daily cycle.

In [ ]:
SID, WEEK = "ETTh2", 15360
WINDOW = window_at(BATCHES, SID, WEEK)
original = WINDOW.data[0, :, 0]  # (168,) °C
assert WEEK >= ENDS[SID] and (original != 0).all()  # evaluation part, none of the runs of zeros ETT has in places
print(f"{SID}, week starting {START + pd.Timedelta(hours=WEEK):%Y-%m-%d}, id {WINDOW.ids[0]}")

**One transformation, one new sample.** A transformation class is built from its parameters, then `transform` takes a batch of windows and returns one `Output` per window: the new window `x`, and everything needed to say what was done to it (`algorithm`, `params`, `seed`, and `parent_id`, the window it comes from).

Here, `drift`: a ramp of +0.11 °C per hour over the week, a **cooling degradation** (the oil warms up for the same load).

In [ ]:
degradation = Drift({"slope": 0.11})
(out,) = degradation.transform(WINDOW)
print("new window:", out.x.shape)
{k: v for k, v in out.row().items() if k in ("id", "parent_id", "algorithm", "params", "seed")}

What happens to the label is a choice of the use case, not of the transformation:

- **forecasting**: the label is the next 24 hours of the same series, inside the signal. To keep it, we transform the history only and read the true future from the record;
- **anomaly detection**: the record has no label at all. A transformation can stay "normal" (harmless variability) or **create** the label "abnormal" (a fault, with its type, size and position, all in `params`).

In [ ]:
def image_panel(ax):
    """Image classification: a picture and its annotation (a drawing, for the comparison)."""
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis("off")
    ax.add_patch(Rectangle((0.6, 0.6), 5.2, 4.6, facecolor="none", edgecolor="black", lw=1.5))
    ax.add_patch(Circle((4.4, 3.9), 0.55, color="C1"))
    ax.add_patch(Polygon([(0.6, 0.6), (2.4, 3.3), (3.5, 2.0), (4.4, 2.8), (5.8, 0.6)], closed=True, color="C0"))
    ax.add_patch(FancyBboxPatch((6.5, 2.4), 3.1, 1.0, boxstyle="round,pad=0.12", facecolor="white", edgecolor="black", lw=1.2))
    ax.text(8.05, 2.9, 'label: "mountain"', ha="center", va="center", fontsize=13)
    ax.plot([5.85, 6.4], [2.9, 2.9], color="black", lw=1.2)
    ax.set_title("One image, one annotation", loc="left")


OT = SERIES[SID]
T = pd.date_range(START, periods=len(OT), freq="h")
fig, axes = plt.subplots(1, 3, figsize=(18.1, 3.3))
image_panel(axes[0])
ax = axes[1]
a, b = WEEK - 96, WEEK + 168 + 96
ax.plot(T[a:b], OT[a:b], color="gray", lw=1, alpha=0.6)
ax.plot(T[WEEK : WEEK + 168], OT[WEEK : WEEK + 168], label="history: 1 week")
ax.plot(T[WEEK + 168 : WEEK + 192], OT[WEEK + 168 : WEEK + 192], lw=2.5, label="next 24 h = label")
ax.xaxis.set_major_locator(mdates.DayLocator(interval=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_ylim(OT[a:b].min() - 3, OT[a:b].max() + 16)
ax.legend(loc="upper left", ncol=2, fontsize=12)
ax.set_title("A window cut out of the series", loc="left")
ax = axes[2]
ax.plot(original, label="recorded")
ax.plot(out.x[:, 0], label="generated: label abnormal, drift +0.11 °C/h")
ax.set_ylim(0, 75)
ax.set_xticks(range(0, 169, 24))
ax.legend(loc="upper left", fontsize=12)
ax.set_title("The same week, with a generated fault", loc="left")
fig.tight_layout(w_pad=2.5)
plt.show()

## 3. Transformations as situations

Each transformation stands for something that happens to a real transformer:

| Transformation | Situation | Role in the detection test |
|---|---|---|
| `jitter`, `sigma` 0.3 °C | noisy temperature probe | normal: must **not** raise an alert |
| `low_pass`, `cutoff` 0.1 cycle/h | readings averaged by the logging system | normal |
| `drift`, `slope` in °C per hour | cooling degradation: the oil warms up for the same load | fault: **must** raise an alert |
| `add_sinusoid`, 1 cycle per 6 h | cooling system cycling on and off | fault |

`jitter` draws random numbers, so it needs a **seed**: each window draws from its own generator, seeded by the seed, the window's id and the parameters, so the same week always gets the same noise, whatever the batch it is in. The other three are deterministic.

In [ ]:
SITUATIONS = {
    "Noisy probe": Jitter({"sigma": 0.3, "seed": 0}),
    "Coarser logging": LowPass({"cutoff": 0.1}),
    "Cooling degradation": Drift({"slope": 0.11}),
    "Cooling cycling": AddSinusoid({"frequency": 0.1666667, "amplitude": 10.0}),  # one cycle every 6 hours
}
SITUATION_OUTPUTS = {title: t.transform(WINDOW)[0] for title, t in SITUATIONS.items()}

fig, axes = plt.subplots(2, 2, figsize=(12.0, 6.5), sharex=True, sharey=True)
for ax, (title, output) in zip(axes.flat, SITUATION_OUTPUTS.items()):
    ax.plot(original, label="recorded")
    ax.plot(output.x[:, 0], label="generated")
    ax.set_title(title, loc="left")
    ax.set_xticks(range(0, 169, 24))
for ax in axes[1]:
    ax.set_xlabel("hour of the week")
for ax in axes[:, 0]:
    ax.set_ylabel("°C")
axes[0, 0].legend(loc="upper right", ncol=2)
fig.tight_layout()
plt.show()

## 4. What a transformation does, exactly

A transformation receives a batch `(B, W, C)` of float windows, with axes `(time, channel)` and no value range, and returns a batch of the same shape. Its parameters are in **cycles per step**, not in hertz: it sees the array, not the sampling rate. ETT is hourly, so the daily cycle is `1/24 ≈ 0.042` cycles per step; the plots show frequencies in cycles per day.

We look at three of them closely, on the followed week: `low_pass` (global: each output hour depends on the whole window), `add_sinusoid` (pointwise, deterministic) and `jitter` (pointwise, random). Each figure shows the week before and after, and its amplitude spectrum. The next window, one day later, shares 144 hours with the followed week: it tells whether two overlapping windows get the same values on the hours they share.

In [ ]:
NEXT_DAY = window_at(BATCHES, SID, WEEK + 24)  # shares hours 24 to 167 of the followed week
LOAD_WINDOW = window_at(LOAD_BATCHES, SID, WEEK)
OIL = {"name": "oil temperature", "unit": "°C", "channels": [6]}
LOAD = {"name": "load", "unit": "HUFL", "channels": [0]}
DAILY_BIN = 7  # 7 cycles in a 168-hour window
FLOOR = 1e-3


def after(transformation, window):
    # The transformed window, as a (168,) array.
    return transformation.transform(window)[0].x[:, 0]


def shared_gap(transformation):
    # Largest difference between the followed week and the next window on their 144 shared hours.
    return np.abs(after(transformation, WINDOW)[24:] - after(transformation, NEXT_DAY)[:144]).max()


def spectrum(week):
    # Single-sided amplitude spectrum, in the channel's units, against frequency in cycles per day.
    amplitude = np.abs(np.fft.rfft(week)) / len(week)
    amplitude[1:] *= 2  # fold the negative frequencies in; 0 (the mean) is not folded
    amplitude[-1] /= 2  # nor is the Nyquist bin (168 hours is even)
    return np.fft.rfftfreq(len(week), 1 / 24), amplitude


def show_transform(before, transformed, label, channel, mark=None):
    # The week before and after: the signal (left) and its amplitude spectrum, log scale (right).
    fig, (ax_t, ax_f) = plt.subplots(1, 2, figsize=(13, 3), gridspec_kw={"width_ratios": [3, 2]})
    for week, lab in [(before, "before"), (transformed, f"after: {label}")]:
        ax_t.plot(week, lw=0.9, label=lab)
        f, amplitude = spectrum(week)
        ax_f.semilogy(f, np.maximum(amplitude, FLOOR), lw=0.9, label=lab)
    if mark is not None:
        ax_f.axvline(mark * 24, color="gray", ls="--", lw=0.8)
    ax_t.set_ylabel(f"{channel['name']} ({channel['unit']})")
    ax_t.set_xlabel("hour of the week")
    ax_t.set_xticks(range(0, 169, 24))
    ax_t.legend(loc="upper right", fontsize=8)
    ax_f.set_ylim(FLOOR / 2, None)
    ax_f.set_ylabel(f"amplitude ({channel['unit']})")
    ax_f.set_xlabel("frequency (cycles per day)")
    fig.suptitle(f"{channel['name']} selection (channels: {channel['channels']})", fontsize=10)
    fig.tight_layout()
    plt.show()

**`low_pass`: frequencies above `cutoff` are set to zero.** It is an ideal FFT filter. A one-week window holds exactly 7 days, so the daily cycle sits on one FFT bin (7 cycles per window), and so do its harmonics (12 hours, 8 hours…): the filter keeps or removes each of them whole.

On the oil temperature, `cutoff = 0.03` cycles per hour, i.e. 0.72 cycles per day (dashed line): the daily cycle is removed, the slow change over the week stays.

In [ ]:
smooth = after(LowPass({"cutoff": 0.03}), WINDOW)
print(f"daily-cycle amplitude: {spectrum(original)[1][DAILY_BIN]:.2f} °C before, {spectrum(smooth)[1][DAILY_BIN]:.1e} after")
show_transform(original, smooth, "low_pass 0.03 (0.72 per day)", OIL, mark=0.03)

On the load selection, `cutoff = 0.06`, i.e. 1.44 cycles per day: the daily cycle stays, its harmonics (2 cycles per day and above) and the hour-to-hour fluctuations are removed.

In [ ]:
load = LOAD_WINDOW.data[0, :, 0]
smooth_load = after(LowPass({"cutoff": 0.06}), LOAD_WINDOW)
(_, before), (_, filtered) = spectrum(load), spectrum(smooth_load)
print(f"load: daily cycle {before[DAILY_BIN]:.2f} -> {filtered[DAILY_BIN]:.2f}; "
      f"12-hour harmonic {before[2 * DAILY_BIN]:.2f} -> {filtered[2 * DAILY_BIN]:.1e}")
show_transform(load, smooth_load, "low_pass 0.06 (1.44 per day)", LOAD, mark=0.06)

Cutoffs of 0.03, 0.06 and 0.09 cycles per hour (0.72, 1.44 and 2.16 per day) sit on either side of one harmonic: 0.03 removes the daily cycle, 0.06 keeps it, 0.09 also keeps the 12-hour harmonic. The band between two cutoffs holds more than the harmonic, though: real data has energy on every bin, so the outputs differ by everything in that band.

And because `low_pass` reads the whole window, two overlapping windows give **two different outputs for the same hours**. The FFT also treats a window as periodic: it sees the jump from the last value back to the first one, and the filter rings at both edges.

In [ ]:
outs = {c: after(LowPass({"cutoff": c}), WINDOW) for c in (0.03, 0.06, 0.09)}
for lo, hi in [(0.03, 0.06), (0.06, 0.09)]:
    print(f"cutoffs {lo} and {hi} differ by up to {np.abs(outs[lo] - outs[hi]).max():.2f} °C")
f, amplitude = spectrum(original)
band = (f > 0.03 * 24) & (f <= 0.06 * 24)
print("bins between 0.03 and 0.06 (cycles per day: °C):", dict(zip(f[band].round(2).tolist(), amplitude[band].round(2).tolist())))
print(f"followed week vs next window, on their 144 shared hours: {shared_gap(LowPass({'cutoff': 0.03})):.2f} °C apart after "
      f"low_pass 0.03, {np.abs(original[24:] - NEXT_DAY.data[0, :144, 0]).max():.0f} before")

**`add_sinusoid`: a wave at a known frequency.** A 6-hour oscillation (`frequency: 1/6`, written `0.1666667` in the configs) of `amplitude: 1` °C. The output minus the input is exactly the requested wave. 6 hours makes 28 whole periods in a week, so the wave sits on one bin: the spectrum shows a peak of about 1 °C there. The phase is counted from each window's first step: two windows one day apart start 4 whole periods apart and add the same values on their shared hours. At 0.1 cycles per hour, 2.4 periods, they do not.

In [ ]:
cycling = AddSinusoid({"frequency": 1 / 6, "amplitude": 1.0})
wavy = after(cycling, WINDOW)
print("output - input is the wave:", np.allclose(wavy - original, np.sin(2 * np.pi * np.arange(168) / 6)))
for frequency in (1 / 6, 0.1):
    gap = shared_gap(AddSinusoid({"frequency": frequency, "amplitude": 1.0}))
    print(f"frequency {frequency:.4g}: followed week and next window differ on their shared hours by up to {gap:.3g} °C")
show_transform(original, wavy, "add_sinusoid every 6 h, amplitude 1 °C", OIL, mark=1 / 6)

**`jitter` and the seed.** `jitter` adds independent Gaussian noise of standard deviation `sigma` to every hour. It draws random numbers, so it **requires** a `seed`, recorded on every output. Each window draws from its own generator, seeded by the seed, its id and the parameters, not by its position in a batch. White noise has the same expected amplitude at every frequency: on the spectrum it raises the floor.

What the seed guarantees, with `sigma = 0.3` °C on the batch that holds the followed week (`apply` takes the windows and one generator per window, from their ids):

In [ ]:
probe = Jitter({"sigma": 0.3, "seed": 0})
batch = next(b for b in BATCHES if WINDOW.ids[0] in b.ids)
xs, ids = batch.data, list(batch.ids)
base = probe.apply(xs, probe.rngs(ids))
noise = base - xs
print(f"noise mean {noise.mean():+.4f}, std {noise.std():.4f} (sigma 0.3)")
print("same seed, same output:     ", np.array_equal(base, probe.apply(xs, probe.rngs(ids))))
reseeded = Jitter({"sigma": 0.3, "seed": 1})
print("other seed, other output:   ", not np.array_equal(base, reseeded.apply(xs, reseeded.rngs(ids))))

# The batch split and the order do not matter: each window keeps its own draw.
one_by_one = np.concatenate([probe.apply(xs[i : i + 1], probe.rngs(ids[i : i + 1])) for i in range(len(xs))])
order = np.random.default_rng(0).permutation(len(xs))
shuffled = probe.apply(xs[order], probe.rngs([ids[i] for i in order]))
print("window by window = batch:   ", np.array_equal(one_by_one, base))
print("shuffled batch, same draws: ", np.array_equal(shuffled, base[order]))
try:
    Jitter({"sigma": 0.3})
except ValueError as error:
    print("no seed:", error)

show_transform(original, after(probe, WINDOW), "jitter, sigma 0.3 °C", OIL)

Each window draws its own noise, so two overlapping windows get **different** noise on the hours they share: one measured hour, jittered twice, differently. The parameters enter the seed too, so two values of `sigma` in a sweep draw **independent** noises, not one noise scaled.

In [ ]:
wider = Jitter({"sigma": 0.6, "seed": 0})
large = wider.apply(xs, wider.rngs(ids)) - xs
print(f"correlation between the sigma=0.3 and sigma=0.6 noises, over the batch: {np.corrcoef(noise.ravel(), large.ravel())[0, 1]:+.3f}")
print(f"followed week and next window, noise on their shared hours differs by up to {shared_gap(probe):.2f} °C")

## 5. Time-series noises

`jitter` is white noise: every hour independent of the others. Real disturbances have a memory. Four noises take time into account, and each one is either **additive** (`x + n`: the same size whatever the level) or **multiplicative** (`x · (1 + n)`: proportional to the level), chosen with `mode`.

| Transformation | Situation | Parameters |
|---|---|---|
| `colored_noise` | drifting probe: pink (1/f) noise, slow wanderings dominate | `sigma` 1 °C, `exponent` 1, additive |
| `ou_noise` | fluctuating offset that comes back (Ornstein-Uhlenbeck), correlated over about 10 hours | `sigma` 1.5 °C, `theta` 0.1, additive |
| `garch_noise` | calm and agitated periods (GARCH volatility), proportional to the temperature | `sigma` 5 %, `alpha` 0.15, `beta` 0.8, multiplicative |
| `impulse_noise` | acquisition glitches: isolated spikes | `rate` 2 % of the hours, `amplitude` ±8 °C, additive |

In [ ]:
NOISES = {
    "Pink noise (1/f), additive": ColoredNoise({"sigma": 1.0, "exponent": 1.0, "seed": 0}),
    "Ornstein-Uhlenbeck, additive": OUNoise({"sigma": 1.5, "theta": 0.1, "seed": 0}),
    "GARCH, multiplicative": GarchNoise({"sigma": 0.05, "alpha": 0.15, "beta": 0.8, "mode": "multiplicative", "seed": 0}),
    "Impulse spikes, additive": ImpulseNoise({"rate": 0.02, "amplitude": 8.0, "seed": 0}),
}
NOISE_OUTPUTS = {title: t.transform(WINDOW)[0] for title, t in NOISES.items()}

fig, axes = plt.subplots(2, 2, figsize=(12.0, 6.5), sharex=True, sharey=True)
for ax, (title, output) in zip(axes.flat, NOISE_OUTPUTS.items()):
    ax.plot(original, label="recorded")
    ax.plot(output.x[:, 0], label="generated")
    ax.set_title(title, loc="left")
    ax.set_xticks(range(0, 169, 24))
for ax in axes[1]:
    ax.set_xlabel("hour of the week")
for ax in axes[:, 0]:
    ax.set_ylabel("°C")
axes[0, 0].legend(loc="upper right", ncol=2)
fig.tight_layout()
plt.show()

## 6. Generated data is labelled data

Every output carries its own record: where it comes from and what was done to it. Written by the pipeline, these records are the rows of the **ledger**. Here are the eight weeks generated above from the spring week. ETT has no labelled failure: we did not need one, the ledger is the label (which fault, of what size, in which week).

In [ ]:
pd.DataFrame([
    {"source window": f"{SID}, hours {WEEK}–{WEEK + 167}", **{k: o.row()[k] for k in ("algorithm", "params", "seed", "parent_id", "id")}}
    for o in [*SITUATION_OUTPUTS.values(), *NOISE_OUTPUTS.values()]
])

## 7. From one week to test sets: sweeping parameters

One week and one parameter value tell a story; a test needs every week, and **a range of sizes** for each degradation and each fault. A sweep is written as a `SweepConfig`, the same block as in the YAML configs: an interval and a number of values. It expands into one transformation per value.

Then `generate` applies every transformation to every window and keeps one record per output: a small ledger, in memory.

In [ ]:
def sweep(make, **sweep_config):
    """One transformation per value of a sweep; ``make`` builds it from the value."""
    return [make(v) for v in SweepConfig(**sweep_config).values()]


#: The parameter each curve is drawn against.
SWEPT = {"jitter": "sigma", "low_pass": "cutoff", "drift": "slope", "add_sinusoid": "amplitude",
         "colored_noise": "sigma", "ou_noise": "sigma", "garch_noise": "sigma", "impulse_noise": "amplitude"}


def generate(batches, transformations):
    """Apply every transformation to every window: the records (a ledger) and the new windows."""
    rows, windows = [], []
    for batch in batches:
        keys = batch.columns.to_pandas()[["series_id", "start_window", "end_window"]].to_dict("records")
        for t in transformations:
            for key, o in zip(keys, t.transform(batch), strict=True):
                rows.append({**key, "algorithm": o.algorithm, "params": o.params, "seed": o.seed, "parent_id": o.parent_id, "id": o.id})
                windows.append(o.x[:, 0])
    ledger = pd.DataFrame(rows)
    ledger["value"] = [p[SWEPT[a]] for a, p in zip(ledger.algorithm, ledger.params)]  # the x axis of the curves
    return ledger, np.stack(windows)


def evaluation(ledger, windows):
    """Keep the generated windows whose source window starts after the training part."""
    keep = (ledger.start_window >= ledger.series_id.map(ENDS)).to_numpy()
    return ledger[keep].reset_index(drop=True), windows[keep]


# Anomaly detection: normal variabilities, among them the time-series noises swept from mild to strong,
# and faults swept over their size.
ANOMALY_TRANSFORMATIONS = [
    Jitter({"sigma": 0.3, "seed": 0}),
    LowPass({"cutoff": 0.1}),
    *sweep(lambda v: ColoredNoise({"sigma": v, "exponent": 1.0, "seed": 0}), range=[0.5, 3.0], samples=6),
    *sweep(lambda v: OUNoise({"sigma": v, "theta": 0.1, "seed": 0}), range=[0.5, 3.0], samples=6),
    *sweep(lambda v: GarchNoise({"sigma": v, "alpha": 0.15, "beta": 0.8, "mode": "multiplicative", "seed": 0}),
           range=[0.02, 0.12], samples=6),
    *sweep(lambda v: Drift({"slope": v}), range=[0.01, 0.16], samples=4),
    *sweep(lambda v: AddSinusoid({"frequency": 0.1666667, "amplitude": v}), range=[1.0, 16.0], samples=6),
    *sweep(lambda v: ImpulseNoise({"rate": 0.02, "amplitude": v, "seed": 0}), range=[2.0, 16.0], samples=8),
]
ANOMALY_ALL, ANOMALY_ALL_WINDOWS = generate(BATCHES, ANOMALY_TRANSFORMATIONS)
ANOMALY, ANOMALY_WINDOWS = evaluation(ANOMALY_ALL, ANOMALY_ALL_WINDOWS)
print(f"{len(ANOMALY_TRANSFORMATIONS)} transformations x {len(KEYS)} windows = {len(ANOMALY_ALL)} generated weeks, "
      f"{len(ANOMALY)} in the evaluation part")
ANOMALY.groupby("algorithm", sort=False).value.unique().to_frame("swept values")

Forecasting degrades the history in two ways, swept: probe noise from 0 to 2 °C, and coarser logging (`low_pass`: the lower the cutoff, the smoother the readings). Same one-week windows: each is the history of a 24-hour forecast.

In [ ]:
FORECAST_TRANSFORMATIONS = [
    *sweep(lambda v: Jitter({"sigma": v, "seed": 0}), range=[0.0, 2.0], samples=5),
    *sweep(lambda v: LowPass({"cutoff": v}), range=[0.05, 0.25], samples=3),
]
FORECAST_ALL, FORECAST_ALL_WINDOWS = generate(BATCHES, FORECAST_TRANSFORMATIONS)
FORECAST, FORECAST_WINDOWS = evaluation(FORECAST_ALL, FORECAST_ALL_WINDOWS)
print(f"{len(FORECAST_TRANSFORMATIONS)} transformations x {len(KEYS)} windows = {len(FORECAST_ALL)} generated histories, "
      f"{len(FORECAST)} in the evaluation part")

## 8. Forecasting: how much does a degraded history cost?

Two forecasters predict the next 24 hours from a one-week history:

- **seasonal naive**: tomorrow is today;
- **lag ridge**: a ridge regression on the week, fitted on clean training windows.

Each generated window is a degraded history; the **true future is read from the original series** after the window's end, so it is never transformed. We measure the mean absolute error of the 24-hour forecast as the degradation grows.

In [ ]:
HISTORIES, FUTURES = em.training_set(SERIES, 168)  # clean (week, next 24 h) pairs from the training part
MODELS = {
    "lag ridge": em.LagRidge().fit(HISTORIES, FUTURES),
    "seasonal naive": em.SeasonalNaive().fit(HISTORIES, FUTURES),
}
ERRORS = em.forecast_errors(FORECAST, FORECAST_WINDOWS, SERIES, MODELS)
TABLE = ERRORS.groupby(["algorithm", "value", "model"]).mae.mean().unstack()
TABLE.round(2)

In [ ]:
PANELS = [
    ("jitter", "Noisy probe", "noise sigma (°C)", False),
    ("low_pass", "Coarser logging", "cutoff (cycles per hour), coarser →", True),
]

fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.2), sharey=True)
for ax, (algorithm, title, xlabel, invert) in zip(axes, PANELS):
    curve = TABLE.loc[algorithm]
    for model in MODELS:
        ax.plot(curve.index, curve[model], marker="o", label=model)
    ax.set_title(title, loc="left")
    ax.set_xlabel(xlabel)
    if invert:
        ax.invert_xaxis()
axes[0].set_ylabel("forecast error, next 24 h (°C)")
axes[0].set_ylim(0, 3.4)
axes[0].legend(loc="lower right")
fig.tight_layout()
plt.show()
clean, noisy = TABLE.loc[("jitter", 0.0), "lag ridge"], TABLE.loc[("jitter", 2.0), "lag ridge"]
print(f"lag ridge: {clean:.2f} °C on clean histories, {noisy:.2f} °C with a 2 °C probe noise ({100 * (noisy / clean - 1):+.0f} %)")

The error grows steadily with the probe noise, for both models. The ridge, the better model on clean data, is also the one that loses most when the logging gets coarser: it relies on the hour-to-hour detail that a smoothed history no longer has.

## 9. Anomaly detection: what does the detector see?

The detector predicts the last 24 hours of each week from the 144 hours before them; the anomaly score is how badly it does. It raises an alert above the 95th percentile of the scores on clean training weeks, so about 5 % of normal weeks raise an alert by construction.

ETT has **no labelled anomaly**. The label comes from the ledger instead: we know which fault was injected, of what size, into which window. The false-alarm rate is measured on the recorded evaluation weeks, straight from the selection.

In [ ]:
DETECTOR = em.SurpriseDetector(window=168).fit(SERIES)
ANOMALY["alert"] = DETECTOR.alerts(ANOMALY_WINDOWS)

RECORDED = np.concatenate([b.data[:, :, 0] for b in BATCHES])  # the 1,438 recorded weeks, in KEYS order
in_evaluation = (KEYS.start_window >= KEYS.series_id.map(ENDS)).to_numpy()
FALSE_ALARMS = DETECTOR.alerts(RECORDED[in_evaluation]).mean() * 100
RATES = ANOMALY.groupby(["algorithm", "value"]).alert.mean() * 100

print(f"alert threshold: {DETECTOR.threshold:.2f} °C mean error")
pd.DataFrame({
    "weeks": ["recorded weeks", "noisy probe (jitter 0.3)", "coarser logging (low_pass 0.1)"],
    "alert rate (%)": [FALSE_ALARMS, RATES.loc[("jitter", 0.3)], RATES.loc[("low_pass", 0.1)]],
}).round(1)

Harmless variability does not raise more alerts than the recorded weeks: the detector is quiet where it should be.

The time-series noises of section 4 play harmless variability too, swept from mild to strong. To compare them on one axis, each is measured by what it adds: the standard deviation, in °C, of the generated week minus the recorded one (GARCH is multiplicative, so its size in °C depends on the temperature).

In [ ]:
NOISE_ROLES = {"colored_noise": "pink noise (1/f)", "ou_noise": "Ornstein-Uhlenbeck", "garch_noise": "GARCH, multiplicative"}
recorded_weeks = np.stack([SERIES[s][w : w + 168] for s, w in zip(ANOMALY.series_id, ANOMALY.start_window)])
ANOMALY["added_std"] = (ANOMALY_WINDOWS - recorded_weeks).std(axis=1)
NOISE_ALARMS = (ANOMALY[ANOMALY.algorithm.isin(list(NOISE_ROLES))]
                .groupby(["algorithm", "value"]).agg(added_std=("added_std", "mean"), alert=("alert", "mean")))
NOISE_ALARMS["alert"] *= 100

fig, ax = plt.subplots(figsize=(12.4, 5.9))
for algorithm, label in NOISE_ROLES.items():
    curve = NOISE_ALARMS.loc[algorithm]
    ax.plot(curve.added_std, curve.alert, marker="o", label=label)
ax.axhline(FALSE_ALARMS, color="gray", lw=1.2, ls="--")
ax.text(NOISE_ALARMS.added_std.max(), FALSE_ALARMS - 1.5, f"false alarms on recorded weeks: {FALSE_ALARMS:.1f} %",
        ha="right", va="top", color="gray", fontsize=13)
ax.set_xlabel("noise added to the week, standard deviation (°C)")
ax.set_ylabel("windows raising an alert (%)")
ax.set_ylim(0, None)
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()
NOISE_ALARMS.round(1)

Up to about 1 °C of added noise, pink and Ornstein-Uhlenbeck noise keep the detector close to its false-alarm rate on recorded weeks (6 % or less, against 4.3 %). GARCH raises alerts sooner, 8.8 % at 1.1 °C, probably because its noise comes in bursts that concentrate the disturbance in a few days. Beyond that, false alarms grow with the noise, whatever its kind: a large disturbance with a memory looks like a surprise. The sweep says how much variability the detector tolerates. Now the faults, from small to large:

In [ ]:
NATURAL_SLOPE = 0.07  # 95th percentile of weekly OT slopes on ETTh2, °C per hour


def fault_figure(rates, xlabel, natural=None, label_high=False):
    fig, ax = plt.subplots(figsize=(12.4, 5.9))
    if natural:
        ax.axvspan(0, natural, color="gray", alpha=0.15, lw=0)
        ax.text(natural / 2, 97, "natural\nweekly slopes", ha="center", va="top", fontsize=13)
    ax.plot(rates.index, rates.values, marker="o")
    ax.axhline(FALSE_ALARMS, color="gray", lw=1.2, ls="--")
    # label_high: above the point labels, when the curve stays close to the dashed line
    label = f"{'dashed line: ' if label_high else ''}false alarms on recorded weeks: {FALSE_ALARMS:.1f} %"
    ax.text(rates.index.max(), FALSE_ALARMS + (14 if label_high else 2.5), label,
            ha="right", va="bottom", color="gray", fontsize=13)
    for x, y in rates.items():
        ax.annotate(f"{y:.0f} %", (x, y), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=13)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("windows raising an alert (%)")
    ax.set_ylim(0, 112)
    ax.set_xticks(rates.index)
    fig.tight_layout()
    plt.show()


fault_figure(RATES.loc["drift"], "drift (°C per hour)", natural=NATURAL_SLOPE)

**Cooling degradation** is caught once it clearly exceeds what the weather and the load already do to the oil temperature: a drift inside the natural range is invisible, a strong one is caught most of the time.

In [ ]:
fault_figure(RATES.loc["add_sinusoid"], "cycling amplitude (°C), one cycle every 6 hours")

**The blind spot.** Cooling cycling is nearly invisible until it is very large. The detector sees the oscillation in the history and *extrapolates* it, instead of being surprised by it: a regular fault is a predictable one. This is a blind spot of every forecast-based detector, and the generated faults reveal it **before a real failure does**.

**Acquisition glitches.** Isolated spikes on 2 % of the hours, swept in amplitude: a fault, the readings are wrong.

In [ ]:
fault_figure(RATES.loc["impulse_noise"], "spike amplitude (°C), on 2 % of the hours", label_high=True)

**A second blind spot.** Even spikes of 16 °C raise an alert on 7 % of the weeks, barely more than the 4.3 % of the recorded weeks. The detector scores the *mean* error over a day, and a few isolated hours weigh little in it: the spikes are averaged away. A detector meant to catch glitches needs a score that looks at single hours, such as the largest error of the day; the generated spikes show the need before a real glitch goes unnoticed.

## 10. The same test sets, from a config

Outside a notebook, nobody writes these loops: a YAML config names the loader and the transformations, and the pipeline (`kcai-data-sampling process -p <config>`, or `run(config)` from Python) writes the ledger as a parquet table and one `.npy` file per generated window. Here is the anomaly config; its `dataloaders` and `transformations` blocks are the ones built by hand above.

In [ ]:
print(Path("examples/config/ett-anomaly.yaml").read_text())

We run the three configs and check that the pipeline wrote **the same windows** as this notebook: same ids, same values. The ids match because they are derived from the same things (the source window, the algorithm, its parameters and seed), and the values match because each window's noise is seeded by its own id.

In [ ]:
NOISES_ALL, NOISES_ALL_WINDOWS = generate(BATCHES, list(NOISES.values()))
IN_NOTEBOOK = {
    "ett-anomaly": (ANOMALY_ALL, ANOMALY_ALL_WINDOWS),
    "ett-forecasting": (FORECAST_ALL, FORECAST_ALL_WINDOWS),
    "ett-noises": (NOISES_ALL, NOISES_ALL_WINDOWS),
}
for name, (ledger, windows) in IN_NOTEBOOK.items():
    with open(f"examples/config/{name}.yaml") as f:
        config = yaml.safe_load(f)
    config["compute"] = {**config.get("compute", {}), "progress_bar": False}  # quiet in a notebook
    shutil.rmtree(f"examples/outputs/{name}", ignore_errors=True)
    run(config)
    root = Path(f"examples/outputs/{name}/ett_ot")
    written = pq.read_table(root / "ledger.parquet").to_pandas()
    mine = dict(zip(ledger.id, windows))
    same = set(written.id) == set(mine) and all(
        np.allclose(np.load(root / "payloads" / a)[:, 0], mine[i]) for i, a in zip(written.id, written.artifact)
    )
    print(f"{name}: {len(written)} rows in the ledger, the same windows as in the notebook: {same}")

written.head(3)  # the ledger of the noises run: one row per generated window

**Each row replays its output.** For a `jitter` row of the anomaly run, we take its source window from the key columns (`series_id`, `start_window`), rebuild the transformation from `algorithm`, `params` and `seed`, and recompute the payload bit for bit.

In [ ]:
root = Path("examples/outputs/ett-anomaly/ett_ot")
anomaly_ledger = pq.read_table(root / "ledger.parquet").to_pandas()
row = anomaly_ledger[anomaly_ledger.algorithm == "jitter"].iloc[3]
source = window_at(BATCHES, row.series_id, row.start_window)
replayed = Jitter({**json.loads(row.params), "seed": int(row.seed)}).transform(source)[0]
print(row[["id", "parent_id", "series_id", "start_window", "end_window", "algorithm", "params", "seed"]].to_frame().T.to_string(index=False))
print("source window found by its key:", source.ids[0] == row.parent_id)
print("replayed from the row, same id:", replayed.id == row.id)
print("replayed from the row, same values:", np.array_equal(np.load(root / "payloads" / row.artifact), replayed.x))

## 11. What this shows

**The interface**

- **A sample is a window**, cut by the loader: one week every day gives 1,438 windows over the two transformers. Its **key** (`series_id`, `start_window`, `end_window`, and `t0_series` when a clock is given) is enough to relate windows: overlap, lag and simultaneity are computed from it, and the ids are opaque hashes, never parsed. Independent windows are drawn reproducibly from a seed.
- **The channel is chosen by the data selection, not by the transformation**: `channels: [0]` and `channels: [6]` give one-channel windows with the same keys and ids.
- **A transformation does exactly what its parameters say.** `low_pass` keeps or removes whole harmonics of a one-week window: at 0.03 cycles per hour the daily cycle of the followed week (4.04 °C) is removed, the slow change stays. `add_sinusoid` adds exactly the requested wave. `jitter` is reproducible and batch-independent: same seed, same output, in any batch split or order; without a seed it is refused.
- **Each window is transformed on its own.** Two overlapping windows can get different values on the hours they share: `low_pass` reads the whole window, `add_sinusoid` counts its phase from the window's start, `jitter` draws per window. The key holds what it would take to make them agree, but the package does not use it yet.

**What it is for**

- **The same transformations serve both use cases.** For forecasting they degrade the input to measure robustness; for detection they play either harmless variability or a fault. The scenario gives the meaning, not the transformation.
- **Every generated window is traceable**: the ledger records its source window, the algorithm, its parameters and seed, enough to replay it bit for bit. That is what turns generated data into labelled test data.
- **Noises with a memory are harmless variability up to a size.** Pink and Ornstein-Uhlenbeck noise up to about 1 °C leave the detector near its false-alarm rate; GARCH, which comes in bursts, raises alerts sooner; larger noises raise false alarms. Which noise is harmless is a choice of the scenario, and the sweep says up to what size the detector agrees.
- **Two blind spots**: regular cooling cycling is extrapolated rather than flagged, and isolated spikes are averaged away by a detector that scores the mean error over a day.
- **Sweeping a parameter gives a curve, not a score**: how fast the forecast degrades, from what size a fault is caught, and where the detector is blind.